In [18]:
#%cd /content/transformer-translation
!git pull

remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 9 (delta 6), reused 9 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 3.11 KiB | 1.56 MiB/s, done.
From https://github.com/hackrudra1234/transformer-translation
   9490a23..732b6f0  main       -> origin/main
Updating 9490a23..732b6f0
Fast-forward
 config.py           |   1 +
 inference/beam.py   | 190 +++++++++++++++++++++++++++++++++++++++++++++++++++-
 inference/greedy.py | 103 +++++++++++++++++++++++++++-
 main_train.py       |  86 ++++++++++++++----------
 training/metrics.py |  75 +++++++++++++++++++++
 5 files changed, 419 insertions(+), 36 deletions(-)


In [3]:
%pip install -q sentencepiece

In [6]:
from datasets import load_dataset

full_dataset = load_dataset(
    "Helsinki-NLP/opus_books",
    "de-en",
    split="train"
)

split_data = full_dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split_data["train"]
validation_dataset = split_data["test"]

print("Train size:", len(train_dataset))
print("Validation size:", len(validation_dataset))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Train size: 46320
Validation size: 5147


In [7]:
from data.bpe_tokenizers import (
    train_bpe_tokenizer,
    load_bpe_tokenizer
)

train_bpe_tokenizer(
    train_dataset=train_dataset,
    model_prefix="data/bpe_shared",
    vocab_size=16000
)

tokenizer = load_bpe_tokenizer(
    "data/bpe_shared.model"
)

sentence = "Es war ganz unmöglich, an diesem Tage einen Spaziergang zu machen."

print(tokenizer.encode(sentence, out_type=str))
print(tokenizer.encode(sentence, out_type=int))

ids = tokenizer.encode(sentence, out_type=int)
print(tokenizer.decode(ids))

BPE tokenizer trained.
Model saved at: data/bpe_shared.model
['▁Es', '▁war', '▁ganz', '▁unmöglich', ',', '▁an', '▁diesem', '▁Tage', '▁einen', '▁Spaziergang', '▁zu', '▁machen', '.']
[542, 131, 383, 3437, 15868, 97, 885, 1719, 338, 15191, 78, 1134, 15871]
Es war ganz unmöglich, an diesem Tage einen Spaziergang zu machen.


In [8]:
test_sentences = [
    "Unwahrscheinlichkeiten sind interessant.",
    "Die Digitalisierung verändert Arbeitsmöglichkeiten.",
    "Internationalisierung ist kompliziert."
]

for sentence in test_sentences:
    print("\nSentence:", sentence)
    print(tokenizer.encode(sentence, out_type=str))


Sentence: Unwahrscheinlichkeiten sind interessant.
['▁Unwahr', 'schein', 'lichkeiten', '▁sind', '▁interessant', '.']

Sentence: Die Digitalisierung verändert Arbeitsmöglichkeiten.
['▁Die', '▁D', 'ig', 'it', 'al', 'is', 'ier', 'ung', '▁verändert', '▁Arbeits', 'möglich', 'keit', 'en', '.']

Sentence: Internationalisierung ist kompliziert.
['▁In', 'tern', 'ational', 'is', 'ier', 'ung', '▁ist', '▁kom', 'p', 'li', 'z', 'iert', '.']


In [10]:
from torch.utils.data import DataLoader

from data.data_utils import create_bpe_collate_fn

In [11]:
bpe_collate_fn = create_bpe_collate_fn(
    tokenizer
)

In [12]:
test_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=bpe_collate_fn
)

In [13]:
batch = next(
    iter(test_loader)
)

In [14]:
print(
    "src:",
    batch["src"].shape
)

print(
    "decoder:",
    batch["decoder_input"].shape
)

print(
    "target:",
    batch["target"].shape
)

print(
    "src mask:",
    batch["src_mask"].shape
)

print(
    "tgt mask:",
    batch["tgt_mask"].shape
)

src: torch.Size([4, 49])
decoder: torch.Size([4, 45])
target: torch.Size([4, 45])
src mask: torch.Size([4, 1, 1, 49])
tgt mask: torch.Size([4, 1, 45, 45])


In [15]:
print(
    batch["src"][0]
)

tensor([    2, 15469,  1298, 12554, 15852,  4845,   999,   690,  8104,   596,
         2220,    97,    52,  5752,  7347,   970, 15868,   639,    52,  6027,
          216,    42,  1012,   105,  5283, 15871,   207,   667, 12554, 15909,
          421, 11433,  5690, 15868,   639,   421, 14118,  7794, 15892,   421,
         2202, 15868,   639,   421, 12649,  1026,   413, 15871,     3])


In [16]:
ids = batch["src"][0].tolist()

ids = [
    x for x in ids
    if x not in [
        tokenizer.pad_id(),
        tokenizer.bos_id(),
        tokenizer.eos_id()
    ]
]

print(
    tokenizer.decode(ids)
)

Solche neu erworbene Staaten sind entweder schon früher an die Herrschaft gewöhnt gewesen, oder die Freiheit ist in ihnen hergebracht. Sie werden erworben: durch fremde Gewalt, oder durch eigne Kräfte; durch Glück, oder durch Tapferkeit.


In [17]:
tokenizer.get_piece_size()

16000

In [19]:
import os

print(os.path.exists("data/bpe_shared.model"))

True


In [20]:
print("Tokenizer:", "SentencePiece BPE")
print("BPE vocab size:", tokenizer.get_piece_size())
print("PAD:", tokenizer.pad_id())
print("UNK:", tokenizer.unk_id())
print("BOS:", tokenizer.bos_id())
print("EOS:", tokenizer.eos_id())
print("Train examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))

Tokenizer: SentencePiece BPE
BPE vocab size: 16000
PAD: 0
UNK: 1
BOS: 2
EOS: 3
Train examples: 46320
Validation examples: 5147


In [ ]:
s